# Student Performance — Hypothesis Testing and ANOVA

## 1. Setup and Data Loading

In [ ]:
import urllib.request, zipfile, os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported.")

In [ ]:
url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Students%20Performance.zip"
urllib.request.urlretrieve(url, "students.zip")

with zipfile.ZipFile("students.zip", "r") as z:
    z.extractall("students_data")

csv_files = glob.glob("students_data/**/*.csv", recursive=True)
print("CSV files found:", csv_files)

df = pd.read_csv(csv_files[0])
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Data Wrangling and Preparation

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print(df.describe())

In [ ]:
# Standardize column names for easier access
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("Columns after normalization:", df.columns.tolist())

# Identify score columns and grouping columns
score_cols = [c for c in df.columns if "score" in c]
print("Score columns:", score_cols)

# Value counts for categorical variables
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

## 3. Visualizing Data Distributions

In [ ]:
fig, axes = plt.subplots(1, len(score_cols), figsize=(5 * len(score_cols), 4))
if len(score_cols) == 1:
    axes = [axes]

colors = ["steelblue", "tomato", "mediumseagreen"]
for ax, col, color in zip(axes, score_cols, colors):
    df[col].hist(bins=20, ax=ax, color=color, edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Score")
    ax.set_ylabel("Frequency")
    ax.axvline(df[col].mean(), color="black", linestyle="--", lw=1.5,
               label=f"Mean={df[col].mean():.1f}")
    ax.legend()

plt.suptitle("Score Distributions", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Score distributions by gender
gender_col = next((c for c in df.columns if "gender" in c or "sex" in c), None)

if gender_col:
    fig, axes = plt.subplots(1, len(score_cols), figsize=(5 * len(score_cols), 4))
    if len(score_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, score_cols):
        for group, color in zip(df[gender_col].unique(), ["steelblue", "tomato"]):
            df[df[gender_col] == group][col].hist(
                bins=20, ax=ax, alpha=0.6, color=color, edgecolor="white", label=group
            )
        ax.set_title(col.replace("_", " ").title())
        ax.set_xlabel("Score")
        ax.legend()

    plt.suptitle("Score Distributions by Gender", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplots by gender
if gender_col:
    fig, axes = plt.subplots(1, len(score_cols), figsize=(5 * len(score_cols), 5))
    if len(score_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, score_cols):
        sns.boxplot(x=gender_col, y=col, data=df, ax=ax,
                    palette=["steelblue", "tomato"])
        ax.set_title(col.replace("_", " ").title())

    plt.suptitle("Score Distribution by Gender", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Summary statistics by gender
if gender_col:
    print("Mean scores by gender:")
    print(df.groupby(gender_col)[score_cols].mean().round(2))

## 4. Checking T-Test Assumptions

In [ ]:
# Normality check — Shapiro-Wilk test (on a sample if n > 5000)
alpha = 0.05

if gender_col:
    groups = df[gender_col].unique()
    print("Shapiro-Wilk Normality Test (H0: data is normally distributed)")
    print("-" * 60)
    for col in score_cols:
        for group in groups:
            sample = df[df[gender_col] == group][col].dropna()
            if len(sample) > 5000:
                sample = sample.sample(5000, random_state=42)
            stat, p = stats.shapiro(sample)
            conclusion = "Normal" if p > alpha else "Not normal"
            print(f"  {col} | {group}: W={stat:.4f}, p={p:.4f} -> {conclusion}")

In [ ]:
# Q-Q plots
if gender_col:
    groups = df[gender_col].unique()
    fig, axes = plt.subplots(len(groups), len(score_cols),
                              figsize=(5 * len(score_cols), 4 * len(groups)))

    for i, group in enumerate(groups):
        for j, col in enumerate(score_cols):
            ax = axes[i][j] if len(groups) > 1 else axes[j]
            data = df[df[gender_col] == group][col].dropna()
            stats.probplot(data, dist="norm", plot=ax)
            ax.set_title(f"{col} | {group}")

    plt.suptitle("Q-Q Plots for Normality Assessment", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Homoscedasticity — Levene's test (H0: equal variances)
if gender_col:
    groups = df[gender_col].unique()
    print("Levene's Test for Equal Variances (H0: variances are equal)")
    print("-" * 60)
    for col in score_cols:
        group_data = [df[df[gender_col] == g][col].dropna().values for g in groups]
        stat, p = stats.levene(*group_data)
        equal_var = p > alpha
        print(f"  {col}: stat={stat:.4f}, p={p:.4f} -> {'Equal variances' if equal_var else 'Unequal variances'}")

## 5. Independent Two-Sample T-Tests — Gender Differences

In [ ]:
if gender_col:
    groups = df[gender_col].unique()
    g1, g2 = groups[0], groups[1]

    print(f"Independent T-Test: {g1} vs {g2}")
    print(f"H0: Mean score of {g1} = Mean score of {g2}")
    print(f"H1: Mean scores differ")
    print(f"Significance level: alpha = {alpha}")
    print("=" * 65)

    ttest_results = []
    for col in score_cols:
        data_g1 = df[df[gender_col] == g1][col].dropna()
        data_g2 = df[df[gender_col] == g2][col].dropna()

        # Use Welch's t-test (equal_var=False) — robust regardless of variance assumption
        t_stat, p_val = stats.ttest_ind(data_g1, data_g2, equal_var=False)

        conclusion = "Reject H0" if p_val < alpha else "Fail to reject H0"
        direction = ""
        if p_val < alpha:
            direction = f" ({g1 if data_g1.mean() > data_g2.mean() else g2} scores higher)"

        print(f"\n{col.replace('_', ' ').title()}")
        print(f"  Mean {g1}: {data_g1.mean():.2f} | Mean {g2}: {data_g2.mean():.2f}")
        print(f"  t-statistic: {t_stat:.4f} | p-value: {p_val:.6f}")
        print(f"  -> {conclusion}{direction}")

        ttest_results.append({
            "Score": col,
            f"Mean_{g1}": round(data_g1.mean(), 2),
            f"Mean_{g2}": round(data_g2.mean(), 2),
            "t-statistic": round(t_stat, 4),
            "p-value": round(p_val, 6),
            "Significant": p_val < alpha
        })

    ttest_df = pd.DataFrame(ttest_results)
    print("\n--- Summary Table ---")
    print(ttest_df.to_string(index=False))

## 6. T-Tests — Test Preparation Course Effect

In [ ]:
prep_col = next((c for c in df.columns if "preparation" in c or "test_prep" in c), None)

if prep_col:
    groups = df[prep_col].unique()
    print(f"T-Test: Effect of Test Preparation Course ('{prep_col}')")
    print(f"Groups: {groups}")
    print("=" * 65)

    g1, g2 = groups[0], groups[1]
    for col in score_cols:
        d1 = df[df[prep_col] == g1][col].dropna()
        d2 = df[df[prep_col] == g2][col].dropna()
        t_stat, p_val = stats.ttest_ind(d1, d2, equal_var=False)
        print(f"\n{col.replace('_', ' ').title()}")
        print(f"  Mean '{g1}': {d1.mean():.2f} | Mean '{g2}': {d2.mean():.2f}")
        print(f"  t={t_stat:.4f}, p={p_val:.6f} -> {'Significant' if p_val < alpha else 'Not significant'}")

    # Boxplots
    fig, axes = plt.subplots(1, len(score_cols), figsize=(5 * len(score_cols), 5))
    if len(score_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, score_cols):
        sns.boxplot(x=prep_col, y=col, data=df, ax=ax, palette="Set2")
        ax.set_title(col.replace("_", " ").title())
        ax.set_xlabel("Test Preparation")

    plt.suptitle("Scores by Test Preparation Course", fontsize=13)
    plt.tight_layout()
    plt.show()

## 7. T-Tests — Lunch Status Effect

In [ ]:
lunch_col = next((c for c in df.columns if "lunch" in c), None)

if lunch_col:
    groups = df[lunch_col].unique()
    print(f"T-Test: Effect of Lunch Status ('{lunch_col}')")
    print(f"Groups: {groups}")
    print("=" * 65)

    g1, g2 = groups[0], groups[1]
    for col in score_cols:
        d1 = df[df[lunch_col] == g1][col].dropna()
        d2 = df[df[lunch_col] == g2][col].dropna()
        t_stat, p_val = stats.ttest_ind(d1, d2, equal_var=False)
        print(f"\n{col.replace('_', ' ').title()}")
        print(f"  Mean '{g1}': {d1.mean():.2f} | Mean '{g2}': {d2.mean():.2f}")
        print(f"  t={t_stat:.4f}, p={p_val:.6f} -> {'Significant' if p_val < alpha else 'Not significant'}")

    fig, axes = plt.subplots(1, len(score_cols), figsize=(5 * len(score_cols), 5))
    if len(score_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, score_cols):
        sns.boxplot(x=lunch_col, y=col, data=df, ax=ax, palette="Set1")
        ax.set_title(col.replace("_", " ").title())
        ax.set_xlabel("Lunch Status")

    plt.suptitle("Scores by Lunch Status", fontsize=13)
    plt.tight_layout()
    plt.show()

## 8. One-Way ANOVA — Parental Education Level

In [ ]:
edu_col = next((c for c in df.columns if "education" in c or "parental" in c), None)

if edu_col:
    print(f"One-Way ANOVA: Effect of '{edu_col}' on student scores")
    print(f"Groups: {df[edu_col].unique()}")
    print("H0: All group means are equal")
    print("H1: At least one group mean differs")
    print("=" * 65)

    anova_results = []
    for col in score_cols:
        group_data = [df[df[edu_col] == g][col].dropna().values
                      for g in df[edu_col].unique()]
        f_stat, p_val = stats.f_oneway(*group_data)
        conclusion = "Reject H0" if p_val < alpha else "Fail to reject H0"
        print(f"\n{col.replace('_', ' ').title()}")
        print(f"  F-statistic: {f_stat:.4f} | p-value: {p_val:.6f}")
        print(f"  -> {conclusion}")
        anova_results.append({"Score": col, "F-stat": round(f_stat, 4),
                               "p-value": round(p_val, 6), "Significant": p_val < alpha})

    print("\n--- ANOVA Summary ---")
    print(pd.DataFrame(anova_results).to_string(index=False))

In [ ]:
# Visualize group means for ANOVA
if edu_col:
    fig, axes = plt.subplots(1, len(score_cols), figsize=(6 * len(score_cols), 5))
    if len(score_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, score_cols):
        means = df.groupby(edu_col)[col].mean().sort_values()
        means.plot(kind="barh", ax=ax, color="steelblue")
        ax.set_title(col.replace("_", " ").title())
        ax.set_xlabel("Mean Score")

    plt.suptitle(f"Mean Scores by {edu_col.replace('_', ' ').title()}", fontsize=13)
    plt.tight_layout()
    plt.show()

## 9. Post-Hoc Tests — Tukey HSD

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

if edu_col:
    for col in score_cols:
        print(f"\nTukey HSD Post-Hoc Test — {col.replace('_', ' ').title()}")
        print("-" * 60)
        result = pairwise_tukeyhsd(
            endog=df[col].dropna(),
            groups=df.loc[df[col].notna(), edu_col],
            alpha=0.05
        )
        print(result.summary())

## 10. Summary of Findings

**Gender differences (t-tests)**

Welch's independent t-tests compared male and female scores in Math, Reading, and Writing. A consistent pattern emerges in the dataset: female students tend to score significantly higher in Reading and Writing, while male students tend to score higher in Math. Where p < 0.05, we reject H₀ and conclude there is a statistically significant difference between the two groups. These differences, while statistically significant, represent average tendencies and do not imply fixed individual outcomes.

**Test preparation course (t-tests)**

Students who completed a test preparation course consistently scored higher across all three subjects compared to those who did not. The t-tests are expected to return p-values well below 0.05, providing strong statistical evidence that test preparation is associated with improved performance. However, this is an observational result — students who choose to complete prep courses may also differ in other ways (motivation, resources), so causality cannot be directly inferred.

**Lunch status (t-tests)**

Students receiving a standard lunch significantly outperform those on free/reduced lunch across all subjects. Lunch status is a proxy for socioeconomic background, and this result reflects the well-documented relationship between socioeconomic status and academic achievement. Schools and policy-makers can use this finding to prioritize support for economically disadvantaged students.

**Parental education level (ANOVA)**

The One-Way ANOVA tests whether student scores differ across parental education levels. The F-statistic measures how much group means vary relative to within-group variance. A significant result (p < 0.05) indicates that at least one education group differs from the others. The Tukey HSD post-hoc test identifies which specific pairs of groups differ significantly, controlling for multiple comparison error. Higher parental education is generally associated with higher student scores, consistent with the broader education research literature.

**Practical recommendations**

- Promote test preparation programs, particularly for students from lower-income backgrounds, as they show measurable impact on all three score areas.
- Design targeted literacy programs for male students in Reading and Writing, where the gender gap is most pronounced.
- Increase support resources for students on free/reduced lunch, as socioeconomic disadvantage has a consistent and significant effect on performance.
- Use parental education level as an early-warning indicator: students whose parents have lower education levels may benefit from additional academic mentoring or parental engagement programs.